## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Ministry of Transport validation, stage 2b: timing in the PM peak and at midday (T8, wave 1b)

**Question.** Stage 2 compared the survey's departure-time profile with the hourly count profile of the cordon links (06:00-19:00) and with the RavKav boarding profile (AM only).
This notebook repeats the coincidence ratio (CR) inside each of the three windows - **AM 06:00-09:00, PM peak 16:00-19:00, midday 10:00-14:00** - and adds the *level*: the share of the
day's 06:00-19:00 traffic that falls in each window, in the survey and on the road. For transit the independent profile is RavKav May 2022 by journey start hour
(`Output/bus/periods/bus_hourly_journeys_2022.csv`, journeys with both ends in the study area), against the survey's bus departures (modes 3 and 5).
**Criterion (guideline 4.1):** CR above 0.6.

In [2]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'
KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(TRIPS, KEYS, 'Input/TAZ_2636_Keys.xlsx')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
_k = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
taz_sz = _k.reindex(TAZ).astype(int)
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ); Z1250 = sorted(taz_1250.unique())
d_ = pd.DataFrame({'z': taz_1250.values, 'g': taz_sz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values}, index=TAZ)
sz_of_1250 = pd.Series({z: int(g.groupby('g')['pop'].sum().idxmax()) for z, g in d_.groupby('z')})

df = pd.read_excel(TRIPS)
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'STStr']).copy()
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636').set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(k26); trips['d1250'] = trips['actTaz'].map(k26)
S = trips[trips['o1250'].isin(Z1250) & trips['d1250'].isin(Z1250)].copy()
S['sz_o'] = S['o1250'].map(sz_of_1250); S['sz_d'] = S['d1250'].map(sz_of_1250)
S['t'] = S['STStr'] % 24; S['w'] = S['new_wf'] / 2
S['grp'] = S['mainmode'].map(lambda m: 'car' if m in (10, 11) else 'bus' if m in (3, 5) else 'metronit' if m == 5 else 'taxi' if m in (4, 8) else 'rail' if m == 7 else 'other')
print(f'survey trips, both ends in the study area, with a start time: {len(S):,} sampled; share of the day departing 06:00-09:00: {S.loc[(S.t >= 6) & (S.t < 9), "w"].sum() / S.w.sum():.3f}')

survey trips, both ends in the study area, with a start time: 106,770 sampled; share of the day departing 06:00-09:00: 0.243


## 1. Car: survey departures against the cordon counts, window by window

In [3]:
CORDONS = {'Haifa city': [8, 12, 13, 14, 15, 16, 17], 'Krayot': [5, 6, 7], 'Kiryat Ata + Zevulun': [9], 'Nazareth + Nof HaGalil': [19, 20],
           'Tirat Carmel': [24], 'Metropolitan core': [5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18]}
WINDOWS = {'AM 06:00-09:00': [6, 7, 8], 'PM 16:00-19:00': [16, 17, 18], 'midday 10:00-14:00': [10, 11, 12, 13]}
cnt = pd.read_csv('Output/validation/car_cordon_count_hourly_profile.csv').replace({'Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)': 'Metropolitan core'})
HOURS_ALL = list(range(6, 20)); hcols = [f'YARAM{h}' for h in HOURS_ALL]
car = S[S.grp == 'car']; rows = []
for name, szs in CORDONS.items():
    o_in = car.sz_o.isin(szs); d_in = car.sz_d.isin(szs)
    for direction, sel in [('inbound', ~o_in & d_in), ('outbound', o_in & ~d_in)]:
        x = car[sel]; sv = np.array([x.loc[(x.t >= h) & (x.t < h + 1), 'w'].sum() for h in HOURS_ALL]); sv = sv / sv.sum()
        cr = cnt[(cnt.cordon == name) & (cnt.direction == direction)]
        if cr.empty: continue
        c = cr[hcols].values.ravel().astype(float); c = c / c.sum()
        for wname, hrs in WINDOWS.items():
            ix = [HOURS_ALL.index(h) for h in hrs]
            rows.append({'cordon': name, 'direction': direction, 'window': wname, 'CR (hours of the window)': vm.coincidence_ratio(sv[ix], c[ix]),
                         'survey share of 06-19': float(sv[ix].sum()), 'count share of 06-19': float(c[ix].sum()),
                         'survey busiest hour share of window': float(sv[ix].max() / sv[ix].sum()), 'count busiest hour share of window': float(c[ix].max() / c[ix].sum())})
car_w = pd.DataFrame(rows); car_w.to_csv(f'{OUT}/T8b_car_profile_by_window.csv', index=False, float_format='%.4f')
agg = car_w.groupby('window').agg(**{'CR min': ('CR (hours of the window)', 'min'), 'CR median': ('CR (hours of the window)', 'median'), 'CR max': ('CR (hours of the window)', 'max'),
                                      'survey share of 06-19 (mean)': ('survey share of 06-19', 'mean'), 'count share of 06-19 (mean)': ('count share of 06-19', 'mean'),
                                      'cordon-directions with CR > 0.6': ('CR (hours of the window)', lambda v: int((v > 0.6).sum()))}).reindex(list(WINDOWS)); print(agg.round(3).to_string())

                    CR min  CR median  CR max  survey share of 06-19 (mean)  count share of 06-19 (mean)  cordon-directions with CR > 0.6
window                                                                                                                                   
AM 06:00-09:00       0.614      0.732   0.821                         0.261                        0.215                               12
PM 16:00-19:00       0.547      0.774   0.965                         0.267                        0.246                               11
midday 10:00-14:00   0.774      0.847   0.967                         0.212                        0.252                               12


In [4]:
for wname in WINDOWS:
    d = car_w[car_w.window == wname]; ok = bool((d['CR (hours of the window)'] > vm.CRIT['CR_min']).all())
    add_summary('T8', '4.1', f'departure profile vs counts, {wname}: 12 cordon-directions', 'cordon, hourly', 'CR > 0.6 in every cordon-direction',
                f'CR {d["CR (hours of the window)"].min():.2f}-{d["CR (hours of the window)"].max():.2f} (median {d["CR (hours of the window)"].median():.2f}); {int((d["CR (hours of the window)"] > .6).sum())} of {len(d)} above 0.6; '
                f'the window holds {d["survey share of 06-19"].mean():.2f} of the survey\'s 06-19 crossings against {d["count share of 06-19"].mean():.2f} on the road (means)',
                vm.verdict(ok, explained=True), 'Shape of the hours inside the window; the share of the day shows whether the survey puts more or less of the day in the window than the road does.', 'survey 2017/18; counts 2017-2023')
fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor='white')
for ax, (wname, hrs) in zip(axes, WINDOWS.items()):
    d = car_w[car_w.window == wname].reset_index(drop=True)
    ax.bar(np.arange(len(d)) - 0.2, d['survey share of 06-19'], 0.4, color=BLUE, label='survey'); ax.bar(np.arange(len(d)) + 0.2, d['count share of 06-19'], 0.4, color=ORANGE, label='counts')
    ax.set_xticks(range(len(d))); ax.set_xticklabels([f"{c[:6]} {dr[:3]}" for c, dr in zip(d.cordon, d.direction)], rotation=75, fontsize=7); ax.legend(frameon=False, fontsize=8)
    style_ax(ax, f'{wname}: share of the 06:00-19:00 crossings', None, 'share')
fig.tight_layout(); save_fig(fig, 'mot_T8b_car_window_shares.png')

## 2. Transit: survey bus departures against RavKav 2022 journeys by start hour

In [5]:
rk = pd.read_csv('Output/bus/periods/bus_hourly_journeys_2022.csv', index_col=0)['journeys, both ends in TAZ_North']
bus = S[S.grp == 'bus']; HR = list(range(5, 23))
sv = np.array([bus.loc[(bus.t >= h) & (bus.t < h + 1), 'w'].sum() for h in HR]); sv = sv / sv.sum(); rv = rk.reindex(HR).fillna(0).values; rv = rv / rv.sum()
rows = [{'window': 'whole day 05:00-23:00', 'CR': vm.coincidence_ratio(sv, rv), 'survey share': 1.0, 'RavKav share': 1.0}]
for wname, hrs in WINDOWS.items():
    ix = [HR.index(h) for h in hrs]
    rows.append({'window': wname, 'CR': vm.coincidence_ratio(sv[ix], rv[ix]), 'survey share': float(sv[ix].sum()), 'RavKav share': float(rv[ix].sum())})
pt = pd.DataFrame(rows); pt.to_csv(f'{OUT}/T8b_bus_profile_by_window.csv', index=False, float_format='%.4f'); print(pt.round(3).to_string(index=False))
prof = pd.DataFrame({'hour': HR, 'survey bus departures (share of 05-23)': sv, 'RavKav 2022 journeys by start hour (share of 05-23)': rv}); prof.to_csv(f'{OUT}/T8b_bus_hourly_profiles.csv', index=False, float_format='%.5f')
for _, r in pt.iterrows():
    add_summary('T8', '4.1', f'bus departure profile vs RavKav 2022 journeys, {r["window"]}', 'hourly', 'CR > 0.6', f'CR {r["CR"]:.2f}; share of the day {r["survey share"]:.2f} (survey) vs {r["RavKav share"]:.2f} (RavKav)',
                vm.verdict(bool(r['CR'] > 0.6), explained=True), 'Departure hour of residents (survey) against the journey start hour of all riders (RavKav); 2017/18 against May 2022.', 'survey 2017/18; RavKav May 2022 (4 Tuesdays)')
fig, ax = plt.subplots(figsize=(8, 4), facecolor='white')
ax.plot(HR, sv, color=BLUE, lw=2, label='survey bus departures'); ax.plot(HR, rv, color=ORANGE, lw=2, label='RavKav 2022 journeys (start hour)')
for wname, hrs in WINDOWS.items(): ax.axvspan(hrs[0], hrs[-1] + 1, color=GRID, alpha=0.5, lw=0)
ax.legend(frameon=False, fontsize=9); style_ax(ax, 'Bus: share of the day by hour (windows shaded)', 'hour', 'share of 05:00-23:00')
save_fig(fig, 'mot_T8b_bus_hourly_profile.png')

               window    CR  survey share  RavKav share
whole day 05:00-23:00 0.820         1.000         1.000
       AM 06:00-09:00 0.732         0.258         0.217
       PM 16:00-19:00 0.844         0.168         0.184
   midday 10:00-14:00 0.876         0.266         0.281


In [6]:
for r in SUMMARY:      # the rows are tagged by the window they describe
    r['period'] = 'PM' if 'PM 16' in r['item'] else 'MD' if 'midday' in r['item'] else ('all day' if 'whole day' in r['item'] else 'AM')

## Summary of this notebook against the guideline

In [7]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage2b.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T8,4.1,"departure profile vs counts, AM 06:00-09:00: 1...","cordon, hourly",CR > 0.6 in every cordon-direction,CR 0.61-0.82 (median 0.73); 12 of 12 above 0.6...,pass
1,T8,4.1,"departure profile vs counts, PM 16:00-19:00: 1...","cordon, hourly",CR > 0.6 in every cordon-direction,CR 0.55-0.96 (median 0.77); 11 of 12 above 0.6...,miss (explained)
2,T8,4.1,"departure profile vs counts, midday 10:00-14:0...","cordon, hourly",CR > 0.6 in every cordon-direction,CR 0.77-0.97 (median 0.85); 12 of 12 above 0.6...,pass
3,T8,4.1,"bus departure profile vs RavKav 2022 journeys,...",hourly,CR > 0.6,CR 0.82; share of the day 1.00 (survey) vs 1.0...,pass
4,T8,4.1,"bus departure profile vs RavKav 2022 journeys,...",hourly,CR > 0.6,CR 0.73; share of the day 0.26 (survey) vs 0.2...,pass
5,T8,4.1,"bus departure profile vs RavKav 2022 journeys,...",hourly,CR > 0.6,CR 0.84; share of the day 0.17 (survey) vs 0.1...,pass
6,T8,4.1,"bus departure profile vs RavKav 2022 journeys,...",hourly,CR > 0.6,CR 0.88; share of the day 0.27 (survey) vs 0.2...,pass
